# DSA4156 Lab: Inmon vs. Kimball on Sample Superstore
## Template notebook

| | |
|---|---|
| **Group name** | *CopiaData* |
| **Members** | *John Adrian Ada, Naijel Sebastian Copia, Bienn Mata* |

You will build two data warehouses from the same source data, in one PostgreSQL database:

- `staging`: raw and cleaned copies of the source files
- `inmon`: a **3NF** enterprise warehouse
- `kimball`: a **denormalized** dimensional model (star or snowflake schema)

**Rules**
- Cells marked `# TODO` are yours to complete. You may add as many cells as you need.
- You may transform with pandas, SQL, or both, but every step must run from this notebook.
- Before submitting, run **Kernel > Restart & Run All** and save the notebook *with outputs*.
- Your data model diagrams go in the PDF report, not here.

## 0. Setup

**Requirements**
- PostgreSQL **15 or newer** (16 recommended). Version 15 is needed for `UNIQUE NULLS NOT DISTINCT`.
- Python packages: `pandas`, `sqlalchemy>=2.0`, `psycopg2-binary`

Create an empty database first, for example in `psql`:
```sql
CREATE DATABASE superstore_dw;
```
Then set your connection string below, or set the `DW_URL` environment variable.
**Do not commit real passwords** in the notebook you submit.

In [ ]:
import os, re
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy import types as sa_types

DATA_DIR = Path("data")
DW_URL = os.getenv("DW_URL", "postgresql+psycopg2://postgres:<password>@localhost:5432/superstore_dw")

engine = create_engine(DW_URL, future=True)

def run_sql(sql: str) -> None:
    """Execute one or more SQL statements (DDL or DML) in a single transaction."""
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)

def q(sql: str, **params) -> pd.DataFrame:
    """Run a SELECT and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

def table_counts(schema: str) -> pd.DataFrame:
    """Row count of every table (partitioned parents included, partitions excluded) in a schema."""
    tables = q("""
        SELECT c.relname AS table_name
        FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname = :s AND c.relkind IN ('r', 'p') AND NOT c.relispartition
        ORDER BY 1""", s=schema)["table_name"]
    rows = [(t, int(q(f'SELECT count(*) AS n FROM "{schema}"."{t}"')["n"][0])) for t in tables]
    return pd.DataFrame(rows, columns=["table_name", "row_count"])

print(q("SELECT version()")["version"][0])

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35228, 64-bit


## 1. Extract

The source system gave us three files (originally three sheets of one workbook):

| File | Contents |
|---|---|
| `orders.csv` | One row per order line (flat, denormalized, 21 columns) |
| `returns.csv` | Orders that were returned |
| `people.csv` | The regional manager of each region |

We read everything **as text** so that nothing is silently converted, then copy the raw data into a
`staging` schema. Staging keeps an untouched copy of the source so the warehouse can always be rebuilt.

In [121]:
raw_orders  = pd.read_csv(DATA_DIR / "orders.csv",  dtype=str, keep_default_na=False)
raw_returns = pd.read_csv(DATA_DIR / "returns.csv", dtype=str, keep_default_na=False)
raw_people  = pd.read_csv(DATA_DIR / "people.csv",  dtype=str, keep_default_na=False)

def snake(col: str) -> str:
    return re.sub(r"[^0-9a-z]+", "_", col.strip().lower()).strip("_")

run_sql("CREATE SCHEMA IF NOT EXISTS staging;")
for name, df in [("orders_raw", raw_orders), ("returns_raw", raw_returns), ("people_raw", raw_people)]:
    df.rename(columns=snake).to_sql(name, engine, schema="staging", if_exists="replace", index=False)

print({n: len(d) for n, d in [("orders", raw_orders), ("returns", raw_returns), ("people", raw_people)]})
raw_orders.head()

{'orders': 9994, 'returns': 800, 'people': 4}


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0,219.582
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


In [122]:
raw_orders['Order ID'].duplicated().sum()

np.int64(4985)

In [123]:
orders = raw_orders.rename(columns=snake)
people = raw_people.rename(columns=snake)
returns_df = raw_returns.rename(columns=snake)

### **Check Data Types and Null Values**

In [124]:
returns_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   returned  800 non-null    object
 1   order_id  800 non-null    object
dtypes: object(2)
memory usage: 12.6+ KB


In [125]:
people.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   regional_manager  4 non-null      object
 1   region            4 non-null      object
dtypes: object(2)
memory usage: 196.0+ bytes


In [126]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   row_id         9994 non-null   object
 1   order_id       9994 non-null   object
 2   order_date     9994 non-null   object
 3   ship_date      9994 non-null   object
 4   ship_mode      9994 non-null   object
 5   customer_id    9994 non-null   object
 6   customer_name  9994 non-null   object
 7   segment        9994 non-null   object
 8   country        9994 non-null   object
 9   city           9994 non-null   object
 10  state          9994 non-null   object
 11  postal_code    9994 non-null   object
 12  region         9994 non-null   object
 13  product_id     9994 non-null   object
 14  category       9994 non-null   object
 15  sub_category   9994 non-null   object
 16  product_name   9994 non-null   object
 17  sales          9994 non-null   object
 18  quantity       9994 non-null

In [127]:
orders.isna().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

While pandas returned no null values, there are still null values for postal code. Further checking shows that they all occur in the rows where the city is Burlington and the state is Vermont. 

In [128]:
len(orders[(orders['city'] == 'Burlington') & (orders['state'] == 'Vermont')])

11

In [129]:
orders[(orders['city'] == 'Burlington') & (orders['state'] == 'Vermont')].T

,2234,5274,8798,9146,9147,9148,9386,9387,9388,9389,9741
row_id,2235,5275,8799,9147,9148,9149,9387,9388,9389,9390,9742
order_id,CA-2018-104066,CA-2016-162887,US-2017-150140,US-2017-165505,US-2017-165505,US-2017-165505,US-2018-127292,US-2018-127292,US-2018-127292,US-2018-127292,CA-2016-117086
order_date,12/5/2018,11/7/2016,4/6/2017,1/23/2017,1/23/2017,1/23/2017,1/19/2018,1/19/2018,1/19/2018,1/19/2018,11/8/2016
ship_date,12/10/2018,11/9/2016,4/10/2017,1/27/2017,1/27/2017,1/27/2017,1/23/2018,1/23/2018,1/23/2018,1/23/2018,11/12/2016
ship_mode,Standard Class,Second Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class
customer_id,QJ-19255,SV-20785,VM-21685,CB-12535,CB-12535,CB-12535,RM-19375,RM-19375,RM-19375,RM-19375,QJ-19255
customer_name,Quincy Jones,Stewart Visinsky,Valerie Mitchum,Claudia Bergmann,Claudia Bergmann,Claudia Bergmann,Raymond Messe,Raymond Messe,Raymond Messe,Raymond Messe,Quincy Jones
segment,Corporate,Consumer,Home Office,Corporate,Corporate,Corporate,Consumer,Consumer,Consumer,Consumer,Corporate
country,United States,United States,United States,United States,United States,United States,United States,United States,United States,United States,United States
city,Burlington,Burlington,Burlington,Burlington,Burlington,Burlington,Burlington,Burlington,Burlington,Burlington,Burlington


We have observed that the null values for postal_code are all from Burlington, Vermont. According to codigo-postal.co, the standard postal code for Burlington, Vermont is 05401, hence we fill the null values with it. 


Reference: https://codigo-postal.co/en-us/usa/vermont/burlington/

In [130]:
orders['postal_code'].fillna('05401', inplace=True)

C:\Users\adaad\AppData\Local\Temp\ipykernel_6840\1312218500.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  orders['postal_code'].fillna('05401', inplace=True)


In [131]:
orders.isna().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

In [132]:
returns_df 

,returned,order_id
0,Yes,CA-2015-100762
1,Yes,CA-2015-100762
2,Yes,CA-2015-100762
3,Yes,CA-2015-100762
4,Yes,CA-2015-100867
...,...,...
795,Yes,US-2018-147886
796,Yes,US-2018-147998
797,Yes,US-2018-151127
798,Yes,US-2018-155999


order_id repeats. This might be due to the fact that an order can contain multiple products, hence when an order is returned, it is recorded as the number of products for that order.

In [133]:
orders[orders['order_id'] == 'CA-2015-100762']

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
6314,6315,CA-2015-100762,11/24/2015,11/29/2015,Standard Class,NG-18355,Nat Gilpin,Corporate,United States,Jackson,...,49201,Central,OFF-AR-10000380,Office Supplies,Art,"Hunt PowerHouse Electric Pencil Sharpener, Blue",151.92,4,0,45.576
6315,6316,CA-2015-100762,11/24/2015,11/29/2015,Standard Class,NG-18355,Nat Gilpin,Corporate,United States,Jackson,...,49201,Central,OFF-LA-10003930,Office Supplies,Labels,"Dot Matrix Printer Tape Reel Labels, White, 50...",196.62,2,0,96.3438
6316,6317,CA-2015-100762,11/24/2015,11/29/2015,Standard Class,NG-18355,Nat Gilpin,Corporate,United States,Jackson,...,49201,Central,OFF-PA-10001815,Office Supplies,Paper,Xerox 1885,144.12,3,0,69.1776
6317,6318,CA-2015-100762,11/24/2015,11/29/2015,Standard Class,NG-18355,Nat Gilpin,Corporate,United States,Jackson,...,49201,Central,OFF-PA-10004082,Office Supplies,Paper,Adams Telephone Message Book w/Frequently-Call...,15.96,2,0,7.98


And indeed, we find that there are four of these orders, just like how the order_id for this order occurs four times in the returns data. We can further check this for all unique values of order_id in the orders table, to see if it occurs the same number of times in the returns table. 

In [134]:
def compare_value_counts(df1, df2, col1, col2, id_name="value"):
    """
    Check whether each unique value in col1 (df1) occurs the same number of
    times as in col2 (df2).

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        The two dataframes to compare.
    col1, col2 : str
        Column names to compare (can differ between df1 and df2).
    id_name : str
        Label to use for the compared column in the output (cosmetic).

    Returns
    -------
    pd.DataFrame
        One row per unique value found in either dataframe, with:
        - count_df1, count_df2 : occurrence counts (0 if missing)
        - match : True if counts are equal
    """
    counts1 = df1[col1].value_counts().rename("count_df1")
    counts2 = df2[col2].value_counts().rename("count_df2")

    comparison = pd.concat([counts1, counts2], axis=1).fillna(0)
    comparison[["count_df1", "count_df2"]] = comparison[["count_df1", "count_df2"]].astype(int)
    comparison["match"] = comparison["count_df1"] == comparison["count_df2"]

    comparison.index.name = id_name
    return comparison.reset_index()

In [135]:
result = compare_value_counts(orders, returns_df, "order_id", "order_id")
result[(result.count_df1 > 0) & (result.count_df2 > 0)]

,value,count_df1,count_df2,match
0,CA-2018-100111,14,14,True
2,CA-2017-165330,11,11,True
10,CA-2016-164882,9,9,True
24,US-2018-118087,8,8,True
28,CA-2018-161956,8,8,True
...,...,...,...,...
4770,CA-2017-136924,1,1,True
4843,CA-2018-154214,1,1,True
4859,CA-2017-148796,1,1,True
4888,CA-2017-105291,1,1,True


Confirming our observations, we drop the duplicates of order_id in returns.

In [136]:
returns_df = returns_df.drop_duplicates(subset='order_id')
returns_df.duplicated().sum()

np.int64(0)

### 1.1 Profile the source: find the functional dependencies

Before designing any model, test which columns determine which. `A -> B` holds when every value of `A`
maps to exactly one value of `B`. The count below is the number of `A` values that break the rule.
These results are the evidence for your normalization decisions in the report.

### **Test Functional Dependencies**

In [137]:
def fd_violations(df: pd.DataFrame, lhs, rhs: str) -> int:
    """Number of distinct LHS values that map to more than one RHS value."""
    counts = df.groupby(lhs, dropna=False)[rhs].nunique(dropna=False)
    return int((counts > 1).sum())

In [138]:
from itertools import permutations
#lhs->rhs

def check_all_fds(df: pd.DataFrame) -> pd.DataFrame:
    columns = list(df.columns)
    results = []

    for lhs, rhs in permutations(columns, 2):
        violations = fd_violations(df, lhs, rhs)
        results.append(
            {
                "Determinant": lhs,
                "Dependent": rhs,
                "Violations": violations,
                "Holds": violations == 0,
            }
        )

    return pd.DataFrame(results, columns=["Determinant", "Dependent", "Violations", "Holds"])

In [139]:
# Example
#print("order_id -> customer_id violations:", fd_violations(orders, "order_id", "customer_id"))

# TODO: test every dependency you think might hold, e.g. customer -> segment, product -> category,
#       postal code -> city, city -> state, state -> region, and so on.
#       Build a small table of results (determinant, dependent, violations) to put in your report.

#testing all columns for orders
pd.set_option("display.max_rows", None)
order_dependency_df = check_all_fds(orders)

In [140]:
order_dependency_df[order_dependency_df["Violations"] > 0].sort_values("Violations", ascending=False)

,Determinant,Dependent,Violations,Holds
20,order_id,row_id,2471,False
39,order_id,profit,2471,False
36,order_id,sales,2471,False
35,order_id,product_name,2469,False
32,order_id,product_id,2469,False
34,order_id,sub_category,2346,False
37,order_id,quantity,2237,False
340,sales,row_id,1986,False
341,sales,order_id,1985,False
343,sales,ship_date,1985,False


In [141]:
#reset display max rows
pd.reset_option("display.max_rows")

### **Check ID Uniqueness in the orders data** 

In [142]:
def check_id_uniqueness(df: pd.DataFrame) -> pd.DataFrame:
    """Check uniqueness of every column whose name contains 'ID'.

    Columns are matched if their name contains 'ID' (case-insensitive). A
    column is considered unique when it has no duplicate non-null values.
    'n duplicates' counts the number of extra occurrences beyond the first,
    i.e. total rows minus the number of distinct non-null values.

    Returns a DataFrame with columns: "column name", "unique", "n duplicates".
    """
    id_cols = [col for col in df.columns if "id" in str(col).lower()]

    results = []
    for col in id_cols:
        non_null = df[col].dropna()
        n_duplicates = int(non_null.duplicated().sum())
        results.append(
            {
                "column name": col,
                "unique": n_duplicates == 0,
                "n duplicates": n_duplicates,
            }
        )

    return pd.DataFrame(
        results, columns=["column name", "unique", "n duplicates"]
    )

In [143]:
check_id_uniqueness(orders)

,column name,unique,n duplicates
0,row_id,True,0
1,order_id,False,4985
2,customer_id,False,9201
3,product_id,False,8132


The criteria for the helper function used is that the column must simply have 'id' in the name, however some columns CAN serve as identifiers without explicitly having 'id' in the name such as 'Region' for a normalized database with a Region table (Region is also present in the 'People' table). For such cases, it is manually checked. 

### **Checking one-to-one mapping between order_id and order_date**

In some case for the orders table/dataframe, for a row with a value of OrderID, A, its Order Date is B such that if A occurs in one row as the ID, its Date is B. It may repeat but its always A-B for any row A or B occurs in. We can verify this below:

In [144]:
n_ids = orders["order_id"].nunique()
n_dates = orders["order_date"].nunique()
n_pairs = len(orders[["order_id", "order_date"]].drop_duplicates())

print(n_ids == n_dates == n_pairs)

False


In [145]:
print(order_dependency_df[(order_dependency_df["Determinant"] == "order_id")&(order_dependency_df["Dependent"] == "order_date")])
print(order_dependency_df[(order_dependency_df["Determinant"] == "order_date")& (order_dependency_df["Dependent"] == "order_id")])

   Determinant   Dependent  Violations  Holds
21    order_id  order_date           0   True
   Determinant Dependent  Violations  Holds
41  order_date  order_id        1014  False


This turns out to be false, or not true both ways. Every order_id maps to exactly one order_date but some dates have more than one order_id hence a many-to-one relationship between order_id and order_date. Altough trivial, its still worth checking for data integrity. 

### **Checking Missing Values**

In [146]:
def report_missing_vals(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().sum()
    report = pd.DataFrame(
        {
            "column": missing.index,
            "missing values": missing.values,
            "percentage": (missing.values / len(df)) * 100,
        }
    )
    return report

In [147]:
report_missing_vals(orders)

,column,missing values,percentage
0,row_id,0,0.0
1,order_id,0,0.0
2,order_date,0,0.0
3,ship_date,0,0.0
4,ship_mode,0,0.0
5,customer_id,0,0.0
6,customer_name,0,0.0
7,segment,0,0.0
8,country,0,0.0
9,city,0,0.0


In [148]:
report_missing_vals(returns_df)

,column,missing values,percentage
0,returned,0,0.0
1,order_id,0,0.0


### **Checking Postal Codes**
By definition (from the hint in the instructions), postal codes must be 5-digits. 

In [149]:
print("Number of non 5-digit postal codes:", len(orders[orders["postal_code"].astype(str).str.len() != 5]))

Number of non 5-digit postal codes: 449


In [150]:
orders[orders["postal_code"].astype(str).str.len() != 5].T

,185,197,267,298,299,300,301,302,306,307,...,9784,9811,9812,9813,9814,9889,9936,9967,9968,9969
row_id,186,198,268,299,300,301,302,303,307,308,...,9785,9812,9813,9814,9815,9890,9937,9968,9969,9970
order_id,CA-2017-105018,CA-2018-107720,CA-2017-111010,CA-2017-142545,CA-2017-142545,CA-2017-142545,CA-2017-142545,CA-2017-142545,CA-2015-111003,CA-2015-111003,...,CA-2016-149748,CA-2017-117583,CA-2017-117583,CA-2017-117583,CA-2017-117583,CA-2018-166499,CA-2018-132955,CA-2018-153871,CA-2018-153871,CA-2018-153871
order_date,11/28/2017,11/6/2018,1/22/2017,10/28/2017,10/28/2017,10/28/2017,10/28/2017,10/28/2017,6/1/2015,6/1/2015,...,5/31/2016,11/27/2017,11/27/2017,11/27/2017,11/27/2017,3/19/2018,6/24/2018,12/11/2018,12/11/2018,12/11/2018
ship_date,12/2/2017,11/13/2018,1/28/2017,11/3/2017,11/3/2017,11/3/2017,11/3/2017,11/3/2017,6/6/2015,6/6/2015,...,6/2/2016,11/30/2017,11/30/2017,11/30/2017,11/30/2017,3/20/2018,6/28/2018,12/17/2018,12/17/2018,12/17/2018
ship_mode,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,Standard Class,...,Second Class,First Class,First Class,First Class,First Class,First Class,Standard Class,Standard Class,Standard Class,Standard Class
customer_id,SK-19990,VM-21685,PG-18895,JD-15895,JD-15895,JD-15895,JD-15895,JD-15895,CR-12625,CR-12625,...,EM-13825,CB-12025,CB-12025,CB-12025,CB-12025,VG-21805,SC-20575,RB-19435,RB-19435,RB-19435
customer_name,Sally Knutson,Valerie Mitchum,Paul Gonzalez,Jonathan Doherty,Jonathan Doherty,Jonathan Doherty,Jonathan Doherty,Jonathan Doherty,Corey Roper,Corey Roper,...,Elizabeth Moffitt,Cassandra Brandow,Cassandra Brandow,Cassandra Brandow,Cassandra Brandow,Vivek Grady,Sonia Cooley,Richard Bierner,Richard Bierner,Richard Bierner
segment,Consumer,Home Office,Consumer,Corporate,Corporate,Corporate,Corporate,Corporate,Home Office,Home Office,...,Corporate,Consumer,Consumer,Consumer,Consumer,Corporate,Consumer,Consumer,Consumer,Consumer
country,United States,United States,United States,United States,United States,United States,United States,United States,United States,United States,...,United States,United States,United States,United States,United States,United States,United States,United States,United States,United States
city,Fairfield,Westfield,Morristown,Belleville,Belleville,Belleville,Belleville,Belleville,Lakewood,Lakewood,...,Paterson,East Orange,East Orange,East Orange,East Orange,Lakewood,Cranston,Plainfield,Plainfield,Plainfield


Upon checking, some of the observations where the postal code is 4 digits simply lack a 0 at the start so 6824 -> 06824, which maps to Conneticut. We apply this rule for all 4 digit zip codes

In [151]:
#add a zero to the prefix of all postal_codes where its length is 4
orders["postal_code"] = orders["postal_code"].astype(str).str.zfill(5)

In [152]:
print("Number of non 5-digit postal codes:",len(orders[orders["postal_code"].astype(str).str.len() != 5]))

Number of non 5-digit postal codes: 0


Next, the distict value counts were checked for the categorical columns, this is useful for the later steps in adding the constraints. For example Region can only be either East, West, Central, or South

In [153]:
#display unique values for categorical columns
print(orders['country'].value_counts())
print('-'*20)
print(orders['ship_mode'].value_counts())
print('-'*20)
print(orders["segment"].value_counts())
print('-'*20)
print(orders["region"].value_counts())
print('-'*20)
print(orders['category'].value_counts())
print('-'*20)
print(orders['sub_category'].value_counts())

country
United States    9994
Name: count, dtype: int64
--------------------
ship_mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64
--------------------
segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64
--------------------
region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64
--------------------
category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64
--------------------
sub_category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64


Checking if quanitity is greater than zero for all rows

In [154]:
len(orders[orders['quantity'].astype(int) > 0]) == len(orders)

True

**Profiling findings** (summarize here, and in more detail in your report)

The orders file is a denormalized, flat table. The column ‘Orders ID’ had 4985 duplicates, suggesting that it cannot be the primary row identifier for this table. It was also observed that the null values for the column for Postal Code are all from Burlington, Vermont. According to codigo-postal.co, the standard postal code for Burlington, Vermont is 05401, hence the null values were filled with it. The length of each postal code entry was also checked, and each entry must be strictly 5-digits. It was found that there are 449 rows where the postal code was not 5-digits. Further checking shows that some of the observations where the postal code is 4 digits simply lack a 0 as the prefix so for example, 6824 is 06824, which maps to Connecticut. This rule was applied for all 4-digit entries, adding a prefix of 0. After applying this, there were no more postal code entries where its digits were not equal to five. Other columns were also checked, such as the Quantity column, which is expected to have all entries be strictly greater than zero. 

The other two tables were also checked for their null values (if there are any), distinct categories, data types and uniqueness. 

## 2. Transform: shared cleaning

Clean the data **once**, then build both warehouses from the same clean staging tables.
That way, any difference between the two warehouses comes from the model, not from the cleaning.

Load your result to `staging.orders_clean`, `staging.returns_clean`, and `staging.people_clean` with proper types.

In [155]:
clean = orders.copy()

In [156]:
# TODO: convert types (dates, numbers), trim text, fix the problems you found in profiling.
#the functions below applies the fixes in the previous step
def clean_orders_table(orders:pd.DataFrame)-> pd.DataFrame:
    # Convert date columns to datetime
    orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')
    orders['ship_date'] = pd.to_datetime(orders['ship_date'], errors='coerce')

    # Convert numeric columns to appropriate types
    orders['quantity'] = pd.to_numeric(orders['quantity'], errors='coerce')
    orders['sales'] = pd.to_numeric(orders['sales'], errors='coerce')
    orders['discount'] = pd.to_numeric(orders['discount'], errors='coerce')
    orders['profit'] = pd.to_numeric(orders['profit'], errors='coerce')

    # Trim whitespace from string columns
    str_cols = orders.select_dtypes(include=['object']).columns
    for col in str_cols:
        orders[col] = orders[col].str.strip()

    # Fix postal code issues (ensure 5 digits)
    orders['postal_code'] = orders['postal_code'].astype(str).str.zfill(5)

    # Check quantity, drop rows where quantity is less than or equal to 0
    if len(orders[orders['quantity'].astype(int) > 0]) != len(orders):
        orders.drop(orders[orders['quantity'].astype(int) <= 0].index, inplace=True)

    # restict segment to either consumer, corporate or home office
    valid_segments = ['Consumer', 'Corporate', 'Home Office']
    orders = orders[orders['segment'].isin(valid_segments)]

    # restrict region to either central, east, south or west
    valid_regions = ['Central', 'East', 'South', 'West']
    orders = orders[orders['region'].isin(valid_regions)]

    #restrict ship_mode to either Standard Class, Second Class, First Class or Same Day
    valid_ship_modes = ['Standard Class', 'Second Class', 'First Class', 'Same Day']
    orders = orders[orders['ship_mode'].isin(valid_ship_modes)]

    # Drop rows with missing critical values (e.g., order_id, product_id)
    orders.dropna(subset=['order_id', 'product_id'], inplace=True)

    return orders

def clean_returns_table(returns_df: pd.DataFrame) -> pd.DataFrame:
    valid_returns = ['Yes', 'No']
    returns_df = returns_df[returns_df['returned'].isin(valid_returns)]

    #map return status to boolean
    returns_df['returned'] = returns_df['returned'].map({'Yes': True, 'No': False})
    returns_df['returned'].astype(bool)

    #drop duplicates
    returns_df = returns_df.drop_duplicates()

    return returns_df 

def clean_people_table(people: pd.DataFrame) -> pd.DataFrame:
    #title case for regional_manager
    people['regional_manager'] = people['regional_manager'].str.title()

    #strip leading whitespaces in regional_manager
    people['regional_manager'] = people['regional_manager'].str.strip()

    #drop duplicates 
    people = people.drop_duplicates()

    #ensure region is within [West, East, Central, South]
    valid_regions = ['West', 'East', 'Central', 'South']
    people = people[people['region'].isin(valid_regions)]

    return people 

In [157]:
returns_clean = raw_returns.rename(columns=snake)   # TODO: clean
people_clean  = raw_people.rename(columns=snake)    # TODO: clean

In [158]:
#apply the cleaning step
clean = clean_orders_table(clean)
returns_clean = clean_returns_table(returns_clean)
people = clean_people_table(people_clean)

In [159]:
# TODO: add assert statements that stop the pipeline if the data is still invalid
def check_orders_data(clean: pd.DataFrame) -> bool:
    try:
        #row_id uniqueness
        assert clean["row_id"].is_unique 

        #check datatypes
        assert pd.api.types.is_datetime64_any_dtype(clean['order_date']) == True 
        assert pd.api.types.is_datetime64_any_dtype(clean['ship_date']) == True
        assert pd.api.types.is_numeric_dtype(clean['quantity']) == True
        assert pd.api.types.is_numeric_dtype(clean['sales']) == True
        assert pd.api.types.is_numeric_dtype(clean['discount']) == True
        assert pd.api.types.is_numeric_dtype(clean['profit']) == True

        #all postal code entries are 5 digits
        assert len(clean['postal_code'] == 5) == len(clean)

        #quantity must be greater than zero for all rows
        assert len(clean[clean['quantity'].astype(int) > 0]) == len(clean)

        #no missing values 
        assert (clean.isna().sum().sum() == 0) and (clean.isnull().sum().sum() == 0)

        #segment must be either consumer, corporate or home office
        valid_segments = ['Consumer', 'Corporate', 'Home Office']
        assert clean['segment'].isin(valid_segments).all()

        #region must be either central, east, south or west
        valid_regions = ['Central', 'East', 'South', 'West']
        assert clean['region'].isin(valid_regions).all()

        #ship_mode must be either Standard Class, Second Class, First Class or Same Day
        valid_ship_modes = ['Standard Class', 'Second Class', 'First Class', 'Same Day']
        assert clean['ship_mode'].isin(valid_ship_modes).all()

    except AssertionError as e:
        print(e)
        return False
    return True 

def check_returns_data(returns_df: pd.DataFrame) -> bool:
    try:
        assert pd.api.types.is_bool_dtype(returns_df['returned'])
        assert returns_df.duplicated().sum().sum() == 0

    except AssertionError as e:
        print(e)
        return False
    return True

def check_people_data(people:pd.DataFrame) -> bool:
    try:
        assert people.duplicated().sum().sum() == 0
        assert people['region'].isin(['West', 'East', 'Central', 'South']).all()
    except AssertionError as e:
        print(e)
        return False
    return True

In [160]:
print("Orders table check: ", check_orders_data(clean))
print("Returns table check: ", check_returns_data(returns_clean))
print("People table check ", check_people_data(people_clean))

Orders table check:  True
Returns table check:  True
People table check  True


In [161]:
clean.to_csv("clean_orders.csv", index=False)
returns_clean.to_csv("returns_clean.csv", index=False)
people_clean.to_csv("people_clean.csv", index=False)

In [162]:
if check_orders_data(clean) and check_returns_data(returns_clean) and check_people_data(people_clean):
    clean.to_sql("orders_clean", engine, schema="staging", if_exists="replace", index=False,
                dtype={"order_date": sa_types.Date, "ship_date": sa_types.Date, 
                       "quantity": sa_types.SmallInteger, "sales": sa_types.Float, 
                       "discount": sa_types.Float, "profit": sa_types.Float, 
                       "postal_code": sa_types.Integer, "row_id": sa_types.Integer})   # TODO: add the other types
    returns_clean.to_sql("returns_clean", engine, schema="staging", if_exists="replace", index=False)
    people_clean.to_sql("people_clean", engine, schema="staging", if_exists="replace", index=False)

    print(table_counts("staging"))
else:
    print("Data is invalid")

      table_name  row_count
0   orders_clean       9994
1     orders_raw       9994
2   people_clean          4
3     people_raw          4
4  returns_clean        296
5    returns_raw        800


## 3. Inmon architecture: 3NF enterprise warehouse

### 3.1 Physical model (DDL)

Write the `CREATE TABLE` statements for your 3NF model. This DDL implements the physical data model in your report.
Use **at least three PostgreSQL-specific features** and explain each one in the report
(for example identity columns, `CHECK` constraints, `UNIQUE NULLS NOT DISTINCT`, indexes, schemas, comments).

One example table is given so you can see the style. Replace or extend it.

In [163]:
INMON_DDL = r"""
DROP SCHEMA IF EXISTS inmon CASCADE;
CREATE SCHEMA inmon;

-- 1. Reference / Lookup Tables (3NF Geographic Hierarchy)
CREATE TABLE inmon.country (
    country_id   INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    country_name VARCHAR(60) NOT NULL UNIQUE
);

CREATE TABLE inmon.region (
    region_id   INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    region_name VARCHAR(20) NOT NULL UNIQUE,
    country_id  INTEGER     NOT NULL REFERENCES inmon.country(country_id)
);

CREATE TABLE inmon.regional_manager (
    manager_id   INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    manager_name VARCHAR(60) NOT NULL,
    region_id    INTEGER     NOT NULL UNIQUE REFERENCES inmon.region(region_id)
);

CREATE TABLE inmon.state (
    state_id   INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    state_name VARCHAR(50) NOT NULL,
    region_id  INTEGER     NOT NULL REFERENCES inmon.region(region_id),
    UNIQUE (state_name, region_id)
);

CREATE TABLE inmon.city (
    city_id   INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    city_name VARCHAR(60) NOT NULL,
    state_id  INTEGER     NOT NULL REFERENCES inmon.state(state_id),
    UNIQUE (city_name, state_id)
);

CREATE TABLE inmon.postal_code (
    postal_code CHAR(5) PRIMARY KEY CHECK (postal_code ~ '^[0-9]{5}$'),
    city_id     INTEGER NOT NULL REFERENCES inmon.city(city_id)
);

-- 2. Customer & Product Classifications
CREATE TABLE inmon.segment (
    segment_id   SMALLINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    segment_name VARCHAR(20) NOT NULL UNIQUE
);

CREATE TABLE inmon.customer (
    customer_id   VARCHAR(10)  PRIMARY KEY,
    customer_name VARCHAR(60)  NOT NULL,
    segment_id    SMALLINT     NOT NULL REFERENCES inmon.segment(segment_id)
);

CREATE TABLE inmon.category (
    category_id   SMALLINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    category_name VARCHAR(30) NOT NULL UNIQUE
);

CREATE TABLE inmon.sub_category (
    sub_category_id   SMALLINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    sub_category_name VARCHAR(30) NOT NULL UNIQUE,
    category_id       SMALLINT NOT NULL REFERENCES inmon.category(category_id)
);

CREATE TABLE inmon.product (
    product_id      VARCHAR(20)  PRIMARY KEY,
    product_name    VARCHAR(150) NOT NULL,
    sub_category_id SMALLINT     NOT NULL REFERENCES inmon.sub_category(sub_category_id)
);

-- 3. Fulfillment & Transactions
CREATE TABLE inmon.ship_mode (
    ship_mode_id   SMALLINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    ship_mode_name VARCHAR(20) NOT NULL UNIQUE
);

CREATE TABLE inmon.sales_order (
    order_id     VARCHAR(20) PRIMARY KEY CHECK (order_id ~ '^[A-Z]{2}-\d{4}-\d{6}$'),
    customer_id  VARCHAR(10) NOT NULL REFERENCES inmon.customer(customer_id),
    order_date   DATE        NOT NULL,
    ship_date    DATE        NOT NULL CHECK (ship_date >= order_date),
    ship_mode_id SMALLINT    NOT NULL REFERENCES inmon.ship_mode(ship_mode_id),
    postal_code  CHAR(5)     NOT NULL REFERENCES inmon.postal_code(postal_code)
);

CREATE TABLE inmon.order_line (
    order_line_id INTEGER PRIMARY KEY, -- Source row_id
    order_id      VARCHAR(20)   NOT NULL REFERENCES inmon.sales_order(order_id),
    product_id    VARCHAR(20)   NOT NULL REFERENCES inmon.product(product_id),
    quantity      SMALLINT      NOT NULL CHECK (quantity > 0),
    sales         NUMERIC(12,4) NOT NULL CHECK (sales >= 0),
    discount      NUMERIC(5,4)  NOT NULL CHECK (discount BETWEEN 0 AND 1),
    profit        NUMERIC(12,4) NOT NULL
);

CREATE TABLE inmon.returned_order (
    order_id VARCHAR(20) PRIMARY KEY REFERENCES inmon.sales_order(order_id),
    returned BOOLEAN     NOT NULL DEFAULT TRUE
);

-- Performance Indexes
CREATE INDEX idx_region_country   ON inmon.region (country_id);
CREATE INDEX idx_state_region     ON inmon.state (region_id);
CREATE INDEX idx_city_state       ON inmon.city (state_id);
CREATE INDEX idx_postal_city      ON inmon.postal_code (city_id);
CREATE INDEX idx_cust_segment     ON inmon.customer (segment_id);
CREATE INDEX idx_subcat_category  ON inmon.sub_category (category_id);
CREATE INDEX idx_prod_subcategory ON inmon.product (sub_category_id);
CREATE INDEX idx_order_customer   ON inmon.sales_order (customer_id);
CREATE INDEX idx_order_postal     ON inmon.sales_order (postal_code);
CREATE INDEX idx_order_ship_mode  ON inmon.sales_order (ship_mode_id);
CREATE INDEX idx_line_order       ON inmon.order_line (order_id);
CREATE INDEX idx_line_product     ON inmon.order_line (product_id);
"""
run_sql(INMON_DDL)

### 3.2 Load the 3NF core

Load parent tables before child tables so that foreign keys resolve.

In [164]:
INMON_LOAD = r"""
-- 1. Country & Regions
INSERT INTO inmon.country (country_name)
SELECT DISTINCT country FROM staging.orders_clean;

INSERT INTO inmon.region (region_name, country_id)
SELECT DISTINCT o.region, c.country_id
FROM staging.orders_clean o
CROSS JOIN inmon.country c;

INSERT INTO inmon.regional_manager (manager_name, region_id)
SELECT p.regional_manager, r.region_id
FROM staging.people_clean p
JOIN inmon.region r ON p.region = r.region_name;

-- 2. State & City
INSERT INTO inmon.state (state_name, region_id)
SELECT DISTINCT o.state, r.region_id
FROM staging.orders_clean o
JOIN inmon.region r ON o.region = r.region_name;

INSERT INTO inmon.city (city_name, state_id)
SELECT DISTINCT o.city, s.state_id
FROM staging.orders_clean o
JOIN inmon.region r ON o.region = r.region_name
JOIN inmon.state s ON o.state = s.state_name AND s.region_id = r.region_id;

-- 3. Postal Code (ensuring 5-digit strings)
INSERT INTO inmon.postal_code (postal_code, city_id)
SELECT DISTINCT ON (LPAD(o.postal_code::text, 5, '0')) 
    LPAD(o.postal_code::text, 5, '0') AS postal_code, 
    c.city_id
FROM staging.orders_clean o
JOIN inmon.state s ON o.state = s.state_name
JOIN inmon.city c ON o.city = c.city_name AND c.state_id = s.state_id
ORDER BY LPAD(o.postal_code::text, 5, '0'), o.city;

-- 4. Customer & Segments
INSERT INTO inmon.segment (segment_name)
SELECT DISTINCT segment FROM staging.orders_clean ORDER BY 1;

INSERT INTO inmon.customer (customer_id, customer_name, segment_id)
SELECT o.customer_id, o.customer_name, s.segment_id
FROM (
    SELECT customer_id, customer_name, segment,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn
    FROM staging.orders_clean
) o
JOIN inmon.segment s ON o.segment = s.segment_name
WHERE o.rn = 1;

-- 5. Product Catalog & Categories
INSERT INTO inmon.category (category_name)
SELECT DISTINCT category FROM staging.orders_clean ORDER BY 1;

INSERT INTO inmon.sub_category (sub_category_name, category_id)
SELECT DISTINCT o.sub_category, c.category_id
FROM staging.orders_clean o
JOIN inmon.category c ON o.category = c.category_name;

INSERT INTO inmon.product (product_id, product_name, sub_category_id)
SELECT o.product_id, o.product_name, sc.sub_category_id
FROM (
    SELECT product_id, product_name, sub_category,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY order_date DESC) AS rn
    FROM staging.orders_clean
) o
JOIN inmon.sub_category sc ON o.sub_category = sc.sub_category_name
WHERE o.rn = 1;

-- 6. Ship Mode
INSERT INTO inmon.ship_mode (ship_mode_name)
SELECT DISTINCT ship_mode FROM staging.orders_clean ORDER BY 1;

-- 7. Sales Order Header
INSERT INTO inmon.sales_order (order_id, customer_id, order_date, ship_date, ship_mode_id, postal_code)
SELECT DISTINCT 
    o.order_id,
    o.customer_id,
    o.order_date,
    o.ship_date,
    sm.ship_mode_id,
    LPAD(o.postal_code::text, 5, '0') AS postal_code
FROM staging.orders_clean o
JOIN inmon.ship_mode sm ON o.ship_mode = sm.ship_mode_name;

-- 8. Order Line
INSERT INTO inmon.order_line (order_line_id, order_id, product_id, quantity, sales, discount, profit)
SELECT 
    row_id,
    order_id,
    product_id,
    quantity,
    sales,
    discount,
    profit
FROM staging.orders_clean;

-- 9. Returned Orders
INSERT INTO inmon.returned_order (order_id, returned)
SELECT DISTINCT r.order_id, TRUE
FROM staging.returns_clean r
WHERE EXISTS (SELECT 1 FROM inmon.sales_order o WHERE o.order_id = r.order_id);
"""
run_sql(INMON_LOAD)
table_counts("inmon")

,table_name,row_count
0,category,3
1,city,604
2,country,1
3,customer,793
4,order_line,9994
5,postal_code,631
6,product,1862
7,region,4
8,regional_manager,4
9,returned_order,296


In [165]:
#sample select first 3 rows from order_line
test = q("SELECT * FROM inmon.customer LIMIT 3")
test

,customer_id,customer_name,segment_id
0,AA-10315,Alex Avila,1
1,AA-10375,Allen Armold,1
2,AA-10480,Andrew Allen,1


In [166]:
checks = {
    "order lines = source rows": int(q("SELECT count(*) n FROM inmon.order_line")["n"][0]) == len(clean),
    "total sales matches": np.isclose(float(q("SELECT sum(sales) s FROM inmon.order_line")["s"][0]), clean["sales"].sum()),
    "total profit matches": np.isclose(float(q("SELECT sum(profit) s FROM inmon.order_line")["s"][0]), clean["profit"].sum()),
}
assert all(checks.values()), checks
checks

{'order lines = source rows': True,
 'total sales matches': np.True_,
 'total profit matches': np.True_}

## 4. Kimball architecture: dimensional model

### 4.1 Declare the model

*TODO: fill in before writing any SQL.*

- **Business process:** Sales Order fulfillment, determining the product sold to which customer, from what location and the profit from it
- **Grain (one fact row = ?):** One fact row is one product line on one order
- **Dimensions:** determined by the characteristics of the grain statement, or what one fact row is. 
- **Facts (measures):** defined by its measures (sales, quantity, profit, discount) and its foreign keys to the dimensions. 

In [167]:
KIMBALL_DDL = r"""
DROP SCHEMA IF EXISTS kimball CASCADE;
CREATE SCHEMA kimball;

-- 1. Date Dimension (Role-Playing)
CREATE TABLE kimball.dim_date (
    date_key     INTEGER     PRIMARY KEY, -- YYYYMMDD
    full_date    DATE        NOT NULL UNIQUE,
    year         SMALLINT    NOT NULL,
    quarter      SMALLINT    NOT NULL CHECK (quarter BETWEEN 1 AND 4),
    month        SMALLINT    NOT NULL CHECK (month BETWEEN 1 AND 12),
    month_name   VARCHAR(10) NOT NULL,
    day_of_month SMALLINT    NOT NULL CHECK (day_of_month BETWEEN 1 AND 31),
    day_of_week  SMALLINT    NOT NULL CHECK (day_of_week BETWEEN 1 AND 7),
    day_name     VARCHAR(10) NOT NULL,
    is_weekend   BOOLEAN     NOT NULL
);

-- 2. Customer Dimension
CREATE TABLE kimball.dim_customer (
    customer_key  INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id   VARCHAR(10)  NOT NULL UNIQUE,
    customer_name VARCHAR(60)  NOT NULL,
    segment       VARCHAR(20)  NOT NULL
);

-- 3. Product Dimension
CREATE TABLE kimball.dim_product (
    product_key   INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    product_id    VARCHAR(20)  NOT NULL UNIQUE,
    product_name  VARCHAR(150) NOT NULL,
    category      VARCHAR(30)  NOT NULL,
    sub_category  VARCHAR(30)  NOT NULL
);

-- 4. Location Dimension (Denormalized)
CREATE TABLE kimball.dim_location (
    location_key     INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    postal_code      CHAR(5)     NOT NULL,
    city             VARCHAR(60) NOT NULL,
    state            VARCHAR(50) NOT NULL,
    region           VARCHAR(20) NOT NULL,
    country          VARCHAR(60) NOT NULL,
    regional_manager VARCHAR(60) NOT NULL,
    UNIQUE (postal_code, city, state)
);

-- 5. Ship Mode Dimension
CREATE TABLE kimball.dim_ship_mode (
    ship_mode_key  SMALLINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    ship_mode_name VARCHAR(20) NOT NULL UNIQUE
);

-- 6. Fact Table
CREATE TABLE kimball.fact_sales (
    sales_key      BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    order_date_key INTEGER       NOT NULL REFERENCES kimball.dim_date(date_key),
    ship_date_key  INTEGER       NOT NULL REFERENCES kimball.dim_date(date_key),
    customer_key   INTEGER       NOT NULL REFERENCES kimball.dim_customer(customer_key),
    product_key    INTEGER       NOT NULL REFERENCES kimball.dim_product(product_key),
    location_key   INTEGER       NOT NULL REFERENCES kimball.dim_location(location_key),
    ship_mode_key  SMALLINT      NOT NULL REFERENCES kimball.dim_ship_mode(ship_mode_key),
    order_id       VARCHAR(20)   NOT NULL,
    returned_flag  BOOLEAN       NOT NULL DEFAULT FALSE,
    sales          NUMERIC(12,4) NOT NULL CHECK (sales >= 0),
    quantity       SMALLINT      NOT NULL CHECK (quantity > 0),
    discount       NUMERIC(5,4)  NOT NULL CHECK (discount BETWEEN 0 AND 1),
    profit         NUMERIC(12,4) NOT NULL
);

CREATE INDEX idx_fact_sales_order_date ON kimball.fact_sales (order_date_key);
CREATE INDEX idx_fact_sales_ship_date  ON kimball.fact_sales (ship_date_key);
CREATE INDEX idx_fact_sales_customer   ON kimball.fact_sales (customer_key);
CREATE INDEX idx_fact_sales_product    ON kimball.fact_sales (product_key);
CREATE INDEX idx_fact_sales_location   ON kimball.fact_sales (location_key);
CREATE INDEX idx_fact_sales_ship_mode  ON kimball.fact_sales (ship_mode_key);
"""
run_sql(KIMBALL_DDL)

### 4.2 Load dimensions, then the fact table

Tip: PostgreSQL's `generate_series()` can build a complete date dimension in a single statement.

In [168]:
KIMBALL_LOAD = r"""
-- 1. Date Dimension via generate_series
INSERT INTO kimball.dim_date (date_key, full_date, year, quarter, month, month_name, day_of_month, day_of_week, day_name, is_weekend)
SELECT 
    TO_CHAR(d, 'YYYYMMDD')::INTEGER AS date_key,
    d::DATE AS full_date,
    EXTRACT(YEAR FROM d)::SMALLINT AS year,
    EXTRACT(QUARTER FROM d)::SMALLINT AS quarter,
    EXTRACT(MONTH FROM d)::SMALLINT AS month,
    TRIM(TO_CHAR(d, 'Month')) AS month_name,
    EXTRACT(DAY FROM d)::SMALLINT AS day_of_month,
    EXTRACT(ISODOW FROM d)::SMALLINT AS day_of_week,
    TRIM(TO_CHAR(d, 'Day')) AS day_name,
    EXTRACT(ISODOW FROM d) IN (6, 7) AS is_weekend
FROM generate_series('2014-01-01'::DATE, '2020-12-31'::DATE, '1 day'::interval) d;

-- 2. Customer Dimension
INSERT INTO kimball.dim_customer (customer_id, customer_name, segment)
SELECT customer_id, customer_name, segment
FROM (
    SELECT customer_id, customer_name, segment,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn
    FROM staging.orders_clean
) o
WHERE o.rn = 1;

-- 3. Product Dimension
INSERT INTO kimball.dim_product (product_id, product_name, category, sub_category)
SELECT product_id, product_name, category, sub_category
FROM (
    SELECT product_id, product_name, category, sub_category,
           ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY order_date DESC) AS rn
    FROM staging.orders_clean
) o
WHERE o.rn = 1;

-- 4. Location Dimension
INSERT INTO kimball.dim_location (postal_code, city, state, region, country, regional_manager)
SELECT DISTINCT 
    LPAD(o.postal_code::text, 5, '0') AS postal_code,
    o.city, 
    o.state, 
    o.region, 
    o.country,
    p.regional_manager
FROM staging.orders_clean o
JOIN staging.people_clean p ON o.region = p.region;

-- 5. Ship Mode Dimension
INSERT INTO kimball.dim_ship_mode (ship_mode_name)
SELECT DISTINCT ship_mode FROM staging.orders_clean ORDER BY 1;

-- 6. Fact Sales Table
INSERT INTO kimball.fact_sales (
    order_date_key,
    ship_date_key,
    customer_key,
    product_key,
    location_key,
    ship_mode_key,
    order_id,
    returned_flag,
    sales,
    quantity,
    discount,
    profit
)
SELECT 
    TO_CHAR(o.order_date, 'YYYYMMDD')::INTEGER AS order_date_key,
    TO_CHAR(o.ship_date, 'YYYYMMDD')::INTEGER AS ship_date_key,
    c.customer_key,
    p.product_key,
    l.location_key,
    sm.ship_mode_key,
    o.order_id,
    CASE WHEN r.order_id IS NOT NULL THEN TRUE ELSE FALSE END AS returned_flag,
    o.sales,
    o.quantity,
    o.discount,
    o.profit
FROM staging.orders_clean o
JOIN kimball.dim_customer c ON o.customer_id = c.customer_id
JOIN kimball.dim_product p ON o.product_id = p.product_id
JOIN kimball.dim_location l ON LPAD(o.postal_code::text, 5, '0') = l.postal_code AND o.city = l.city AND o.state = l.state
JOIN kimball.dim_ship_mode sm ON o.ship_mode = sm.ship_mode_name
LEFT JOIN (SELECT DISTINCT order_id FROM staging.returns_clean) r ON o.order_id = r.order_id;
"""
run_sql(KIMBALL_LOAD)
table_counts("kimball")

,table_name,row_count
0,dim_customer,793
1,dim_date,2557
2,dim_location,632
3,dim_product,1862
4,dim_ship_mode,4
5,fact_sales,9994


In [169]:
# TODO: reconciliation checks for the Kimball warehouse (row count, total sales, total profit)
checks_kimball = {
    "fact rows = source rows": int(q("SELECT count(*) n FROM kimball.fact_sales")["n"][0]) == len(clean),
    "total sales matches": np.isclose(float(q("SELECT sum(sales) s FROM kimball.fact_sales")["s"][0]), clean["sales"].sum()),
    "total profit matches": np.isclose(float(q("SELECT sum(profit) s FROM kimball.fact_sales")["s"][0]), clean["profit"].sum()),
}
assert all(checks_kimball.values()), checks_kimball
checks_kimball

{'fact rows = source rows': True,
 'total sales matches': np.True_,
 'total profit matches': np.True_}

In [170]:
#select a few rows from fact_sales
k_test = q("SELECT * FROM kimball.dim_product LIMIT 5")
k_test

,product_key,product_id,product_name,category,sub_category
0,1,FUR-BO-10000112,"Bush Birmingham Collection Bookcase, Dark Cherry",Furniture,Bookcases
1,2,FUR-BO-10000330,"Sauder Camden County Barrister Bookcase, Plank...",Furniture,Bookcases
2,3,FUR-BO-10000362,Sauder Inglewood Library Bookcases,Furniture,Bookcases
3,4,FUR-BO-10000468,O'Sullivan 2-Shelf Heavy-Duty Bookcases,Furniture,Bookcases
4,5,FUR-BO-10000711,"Hon Metal Bookcases, Gray",Furniture,Bookcases


## 5. Same business questions, both warehouses

Answer each question **twice**: once against the Inmon 3NF warehouse (`inmon` schema) and once against
the Kimball model. Both answers must be identical, and they must match the source data.

1. Total sales and profit by region for each year.
2. Return rate (percentage of order lines that belong to a returned order) and returned sales, by product category.
3. Top 5 customers by sales in 2018, with their segment and number of orders.

In [171]:
Q1_INMON = """
SELECT EXTRACT(YEAR FROM so.order_date)::int      AS year,
       r.region_name                              AS region,
       ROUND(SUM(ol.sales), 2)::float8            AS total_sales,
       ROUND(SUM(ol.profit), 2)::float8           AS total_profit
FROM inmon.order_line ol
JOIN inmon.sales_order so ON so.order_id    = ol.order_id
JOIN inmon.postal_code pc ON pc.postal_code = so.postal_code
JOIN inmon.city        ci ON ci.city_id     = pc.city_id
JOIN inmon.state       st ON st.state_id    = ci.state_id
JOIN inmon.region      r  ON r.region_id    = st.region_id
GROUP BY 1, 2
ORDER BY 1, 2;
"""

Q1_KIMBALL = """
SELECT d.year::int                        AS year,
       l.region                           AS region,
       ROUND(SUM(f.sales), 2)::float8     AS total_sales,
       ROUND(SUM(f.profit), 2)::float8    AS total_profit
FROM kimball.fact_sales f
JOIN kimball.dim_date     d ON d.date_key     = f.order_date_key
JOIN kimball.dim_location l ON l.location_key = f.location_key
GROUP BY 1, 2
ORDER BY 1, 2;
"""
q1_i, q1_k = q(Q1_INMON), q(Q1_KIMBALL)
pd.testing.assert_frame_equal(q1_i, q1_k, check_dtype=False, atol=0.01)

# Kimball vs pandas on the cleaned source
q1_p = (clean.assign(year=clean["order_date"].dt.year)
             .groupby(["year", "region"], as_index=False)
             .agg(total_sales=("sales", "sum"), total_profit=("profit", "sum"))
             .sort_values(["year", "region"]).reset_index(drop=True))
pd.testing.assert_frame_equal(q1_k, q1_p.round(2), check_dtype=False, atol=0.01)

In [172]:
q1_i.head(5)

,year,region,total_sales,total_profit
0,2015,Central,103838.16,539.55
1,2015,East,128680.46,17059.61
2,2015,South,103845.84,11879.12
3,2015,West,147883.03,20065.69
4,2016,Central,102874.22,11716.80


In [173]:
q1_k.head(5)

,year,region,total_sales,total_profit
0,2015,Central,103838.16,539.55
1,2015,East,128680.46,17059.61
2,2015,South,103845.84,11879.12
3,2015,West,147883.03,20065.69
4,2016,Central,102874.22,11716.80


In [174]:
q1_i.equals(q1_k)

True

In [175]:
Q2_INMON = """
SELECT c.category_name                                                        AS category,
       COUNT(*)                                                               AS order_lines,
       COUNT(ro.order_id)                                                     AS returned_lines,
       ROUND(100.0 * COUNT(ro.order_id) / COUNT(*), 2)::float8                AS return_rate_pct,
       ROUND(COALESCE(SUM(ol.sales) FILTER (WHERE ro.order_id IS NOT NULL), 0), 2)::float8
                                                                              AS returned_sales
FROM inmon.order_line ol
JOIN inmon.product      p  ON p.product_id       = ol.product_id
JOIN inmon.sub_category sc ON sc.sub_category_id = p.sub_category_id
JOIN inmon.category     c  ON c.category_id      = sc.category_id
LEFT JOIN inmon.returned_order ro ON ro.order_id = ol.order_id AND ro.returned
GROUP BY c.category_name
ORDER BY c.category_name;
"""

Q2_KIMBALL = """
SELECT p.category                                                             AS category,
       COUNT(*)                                                               AS order_lines,
       COUNT(*) FILTER (WHERE f.returned_flag)                                AS returned_lines,
       ROUND(100.0 * COUNT(*) FILTER (WHERE f.returned_flag) / COUNT(*), 2)::float8
                                                                              AS return_rate_pct,
       ROUND(COALESCE(SUM(f.sales) FILTER (WHERE f.returned_flag), 0), 2)::float8
                                                                              AS returned_sales
FROM kimball.fact_sales f
JOIN kimball.dim_product p ON p.product_key = f.product_key
GROUP BY p.category
ORDER BY p.category;
"""
q2_i, q2_k = q(Q2_INMON), q(Q2_KIMBALL)

pd.testing.assert_frame_equal(q2_i, q2_k, check_dtype=False, atol=0.01)

returned_ids = set(returns_clean.loc[returns_clean["returned"], "order_id"])
tmp = clean.assign(is_returned=clean["order_id"].isin(returned_ids))
tmp["returned_sales"] = tmp["sales"].where(tmp["is_returned"], 0.0)
q2_p = (tmp.groupby("category")
           .agg(order_lines=("order_id", "size"),
                returned_lines=("is_returned", "sum"),
                returned_sales=("returned_sales", "sum"))
           .reset_index())
q2_p["return_rate_pct"] = 100.0 * q2_p["returned_lines"] / q2_p["order_lines"]
q2_p = q2_p[["category", "order_lines", "returned_lines", "return_rate_pct", "returned_sales"]].round(2)
pd.testing.assert_frame_equal(q2_k, q2_p, check_dtype=False, atol=0.01)

In [176]:
q2_k == q2_p

,category,order_lines,returned_lines,return_rate_pct,returned_sales
0,True,True,True,True,True
1,True,True,True,True,True
2,True,True,True,True,True


In [177]:
q2_k

,category,order_lines,returned_lines,return_rate_pct,returned_sales
0,Furniture,2121,171,8.06,59219.17
1,Office Supplies,6026,473,7.85,48576.93
2,Technology,1847,156,8.45,72708.17


In [178]:
q2_p

,category,order_lines,returned_lines,return_rate_pct,returned_sales
0,Furniture,2121,171,8.06,59219.17
1,Office Supplies,6026,473,7.85,48576.93
2,Technology,1847,156,8.45,72708.17


In [179]:
q2_k.equals(q2_p)

True

In [180]:
Q3_INMON = """
SELECT cu.customer_id,
       cu.customer_name,
       sg.segment_name                         AS segment,
       ROUND(SUM(ol.sales), 2)::float8         AS total_sales,
       COUNT(DISTINCT so.order_id)             AS num_orders
FROM inmon.order_line ol
JOIN inmon.sales_order so ON so.order_id    = ol.order_id
JOIN inmon.customer    cu ON cu.customer_id = so.customer_id
JOIN inmon.segment     sg ON sg.segment_id  = cu.segment_id
WHERE so.order_date >= DATE '2018-01-01' AND so.order_date < DATE '2019-01-01'
GROUP BY cu.customer_id, cu.customer_name, sg.segment_name
ORDER BY SUM(ol.sales) DESC, cu.customer_id
LIMIT 5;
"""

Q3_KIMBALL = """
SELECT c.customer_id,
       c.customer_name,
       c.segment                               AS segment,
       ROUND(SUM(f.sales), 2)::float8          AS total_sales,
       COUNT(DISTINCT f.order_id)              AS num_orders
FROM kimball.fact_sales f
JOIN kimball.dim_customer c ON c.customer_key = f.customer_key
JOIN kimball.dim_date     d ON d.date_key     = f.order_date_key
WHERE d.year = 2018
GROUP BY c.customer_id, c.customer_name, c.segment
ORDER BY SUM(f.sales) DESC, c.customer_id
LIMIT 5;
"""
q3_i, q3_k = q(Q3_INMON), q(Q3_KIMBALL)

pd.testing.assert_frame_equal(q3_i, q3_k, check_dtype=False, atol=0.01)

c18 = clean[clean["order_date"].dt.year == 2018]
q3_p = (c18.groupby(["customer_id", "customer_name", "segment"], as_index=False)
           .agg(total_sales=("sales", "sum"), num_orders=("order_id", "nunique"))
           .sort_values(["total_sales", "customer_id"], ascending=[False, True])
           .head(5).reset_index(drop=True).round(2))
pd.testing.assert_frame_equal(q3_k, q3_p, check_dtype=False, atol=0.01)

In [181]:
q3_k == q3_p

,customer_id,customer_name,segment,total_sales,num_orders
0,True,True,True,True,True
1,True,True,True,True,True
2,True,True,True,True,True
3,True,True,True,True,True
4,True,True,True,True,True


In [182]:
q3_p

,customer_id,customer_name,segment,total_sales,num_orders
0,RB-19360,Raymond Buch,Consumer,14203.28,3
1,TA-21385,Tom Ashbrook,Home Office,13723.50,2
2,HL-15040,Hunter Lopez,Consumer,10522.55,2
3,SV-20365,Seth Vernon,Consumer,8459.94,4
4,GT-14635,Grant Thornton,Corporate,8167.42,1


In [183]:
q3_k.equals(q3_p)

True

In [184]:
def n_tables(schema: str) -> int:
    df = q("""
        SELECT COUNT(*) AS n
        FROM information_schema.tables
        WHERE table_schema = :s AND table_type = 'BASE TABLE'
    """, s=schema)
    return int(df["n"][0])

print("Inmon:  ", n_tables("inmon"))
print("Kimball:", n_tables("kimball"))

Inmon:   15
Kimball: 6


In [185]:
def join_count(sql): return len(re.findall(r"\bJOIN\b", sql, flags=re.I))

comparison = pd.DataFrame({
    "question": ["Q1", "Q2", "Q3"],
    "joins_inmon_3NF": [join_count(s) for s in (Q1_INMON, Q2_INMON, Q3_INMON)],
    "joins_kimball": [join_count(s) for s in (Q1_KIMBALL, Q2_KIMBALL, Q3_KIMBALL)],
})
comparison

,question,joins_inmon_3NF,joins_kimball
0,Q1,5,2
1,Q2,4,1
2,Q3,3,2


In [186]:
def schema_size_kb(schema: str) -> dict:
    r = q("""
        SELECT SUM(pg_relation_size(c.oid))       AS table_bytes,
               SUM(pg_indexes_size(c.oid))        AS index_bytes,
               SUM(pg_total_relation_size(c.oid)) AS total_bytes
        FROM pg_class c
        JOIN pg_namespace n ON n.oid = c.relnamespace
        WHERE n.nspname = :s AND c.relkind = 'r'
    """, s=schema).iloc[0]
    return {k.replace("_bytes", "_kb"): round(float(v) / 1024, 1) for k, v in r.items()}

pd.DataFrame({"inmon": schema_size_kb("inmon"), "kimball": schema_size_kb("kimball")})


,inmon,kimball
table_kb,1568.0,1496.0
index_kb,1856.0,1552.0
total_kb,3592.0,3168.0


## 6. Evidence for the report

Export the tables you will show in your report. Screenshots of query results are also fine.
For the ER diagrams of the *resulting* warehouses, you can generate them from the live database with
pgAdmin (ERD Tool), DBeaver, or DataGrip.

In [187]:
out = Path("report_outputs"); out.mkdir(exist_ok=True)
table_counts("inmon").to_csv(out / "inmon_row_counts.csv", index=False)
table_counts("kimball").to_csv(out / "kimball_row_counts.csv", index=False)
comparison.to_csv(out / "join_comparison.csv", index=False)
# TODO: export anything else you need (sample rows, query results, profiling table)
sorted(p.name for p in out.iterdir())

['inmon_row_counts.csv', 'join_comparison.csv', 'kimball_row_counts.csv']